# Week 2 - PART 2 - Agents - SQL Agent Geliştirme Rehberi


## 0. Giriş

Kurumsal sistemlerde verilerin büyük bölümü SQL veritabanlarında tutulur.

Bu nedenle yıllardır Tableau, Power BI, Looker gibi Business Intelligence (BI) araçları kullanılmaktadır.

Peki kullanıcı SQL bilmeden şu soruları sorabilseydi?

- Geçen ay en çok satış yapılan ürün neydi?
- En çok harcama yapan müşteri kim?
- Son 30 gündeki sipariş sayısı nedir?
- Hangi ülkeden daha fazla gelir elde ediyoruz?

Günümüzde LLM'ler (Large Language Models) sayesinde bu mümkün hale geldi.

Kullanıcı doğal dilde soru sorar.

LLM bu soruyu SQL'e çevirir.

Veritabanı sorgulanır.

Sonuç tekrar doğal dilde kullanıcıya sunulur.

Ancak pratikte bu süreç göründüğü kadar kolay değildir.



## 1. SQL Agent Oluştururken Karşılaşılan Problemler
### Problem 1: Hallucination (Hayal Görme)

LLM'ler SQL yazabilir.

Ancak çoğu zaman:

- Olmayan tablolar üretir.
- Olmayan kolonlar üretir.
- Hatalı JOIN ilişkileri kurar.
- Veritabanında bulunmayan alanları kullanır.

**Örneğin:**

```sql
SELECT customer_name
FROM customers
```

LLM böyle bir sorgu üretebilir.

Ancak veritabanında:

`CustomerName`

kolonu varsa sorgu başarısız olur.

Bu nedenle LLM'in gerçek veritabanı yapısını bilmesi gerekir.

---

### Problem 2: Context Window Limiti

Bir veritabanında:

- Yüzlerce tablo
- Binlerce kolon
- Milyonlarca kayıt

olabilir.

Bunların tamamını prompt içine koyamayız.

**Örneğin:**

```
500 tablo
20.000 kolon
```

bilgisini modele göndermek mümkün değildir.

Bu nedenle modele sadece gerekli bilgileri vermemiz gerekir.

---

### Problem 3: LLM Hata Yapabilir

Doğru tabloyu biliyor olabilir.

Doğru kolonları biliyor olabilir.

Yine de yanlış SQL üretebilir.

**Örneğin:**

```sql
SELECT *
FROM Customer
WHERE Total > 100
```

Fakat `Total` kolonu `Customer` tablosunda olmayabilir.

Bu durumda ne yapacağız?

Sistemin hata mesajını okuyup sorguyu düzeltebilmesi gerekir.

---

## 3. İnsanlar Bu Problemleri Nasıl Çözüyor?

Bir veri analistini düşünelim.

Kendisine şu soru soruluyor:

> En çok satış yapan sanatçı kim?

Analist doğrudan SQL yazmaz.

Önce:

- Tablolara bakar.
- Şemayı inceler.
- Örnek satırları görüntüler.
- Küçük test sorguları çalıştırır.
- Hata alırsa düzeltir.

SQL Agent'ın da aynı davranışı göstermesi gerekir.


## 4. Veritabanını LLM'e Nasıl Anlatırız?

LLM'in doğru SQL yazabilmesi için veritabanını tanıtmalıyız.

Bunu üç seviyede yapabiliriz:

1. Şema (Schema)
2. Örnek Veri
3. Örnek Sorgular

## 5. Şema Bilgisini Vermek

En basit yöntem:

Track
- TrackId
- Name
- AlbumId
- GenreId

Ancak araştırmalar bunun yeterli olmadığını gösteriyor.

Daha başarılı yöntem:

```sql
CREATE TABLE Track (
    TrackId INTEGER,
    Name NVARCHAR(200),
    AlbumId INTEGER,
    GenreId INTEGER
)
```

şeklinde gerçek `CREATE TABLE` bilgisini vermektir.

## 6. Neden CREATE TABLE Daha Başarılı?

Çünkü model:

- Kolonları görür.
- Veri tiplerini görür.
- Primary Key'leri görür.
- Foreign Key'leri görür.

**Örneğin:**

```
FOREIGN KEY(GenreId)
REFERENCES Genre(GenreId)
```

satırını gören model JOIN işlemlerini daha doğru kurabilir.



## 7. Örnek Veri Vermek

Sadece şema yeterli değildir.

Model verinin nasıl göründüğünü de bilmelidir.

**Örneğin:**

`Composer`

kolonunu düşünelim.

Bu kolon:

`Mozart`

şeklinde mi?

Yoksa:

`Wolfgang Amadeus Mozart`

şeklinde mi?

Yoksa:

`W. A. Mozart`

şeklinde mi?

Bunu bilmesi gerekir.

Bu nedenle `CREATE TABLE` sonrasında birkaç örnek satır verilmesi performansı artırır.

**Örnek:**

```sql
SELECT * FROM Track LIMIT 3;
```

Sonuç:
```
1 | Song A | Mozart
2 | Song B | Beethoven
3 | Song C | Bach
```

## 8. Kaç Satır Gösterilmeli?

Araştırmalar göstermiştir ki:

3 örnek satır en iyi sonucu vermektedir.

Daha fazla veri göndermek her zaman daha iyi değildir.

Hatta performansı düşürebilir.

**Önerilen yapı:**

`CREATE TABLE ...`

**sonrasında:**

`SELECT * LIMIT 3`

çıktısı vermektir.

## 9. Custom Table Description Kullanımı

Bazı durumlarda ilk 3 satır yeterince açıklayıcı olmayabilir.

**Örneğin:**

`Composer`

alanında bazı kayıtlar:

`Berry Gordy, Jr./Janie Bradford`

şeklinde slash kullanıyor olabilir.

Bu bilgi ilk satırlarda bulunmuyorsa LLM bunu öğrenemez.

Bu nedenle manuel örnekler ekleyebiliriz.

**Örnek:**
```
1 | Song A | Mozart
2 | Song B | Beethoven
3 | Song C | Bach
4 | Song D | Berry Gordy/Janie Bradford
```

## 10. Hassas Verileri Gizlemek

Gerçek müşteri verilerini modele göndermek istemeyebilirsiniz.

Bu durumda:

Gerçek veri yerine örnek veri verebilirsiniz.

**Örneğin:**

Gerçek:

`Ahmet Yılmaz`

**Yerine:**

`Customer A`

kullanabilirsiniz.

## 11. Çıktı Boyutunu Sınırlandırmak

SQL Agent geliştirirken en kritik konulardan biri budur.

**Kötü örnek:**

```sql
SELECT *
FROM Orders
````

Bu sorgu:

- 100.000 satır döndürebilir.
- Context Window'u doldurabilir.
- Maliyet artırabilir.

**Daha doğru yaklaşım:**

```
SELECT
    OrderId,
    Total
FROM Orders
LIMIT 10
```

## 12. top_k Kullanımı

LangChain SQL Agent'larında genellikle:

`top_k = 10`

kullanılır.

Böylece model:

`LIMIT 10`

eklemeye teşvik edilir.

**Örnek:**

```sql
SELECT
    Country,
    SUM(Total)
FROM Invoice
GROUP BY Country
ORDER BY SUM(Total) DESC
LIMIT 10
```

## 13. SQL Hatalarını Otomatik Düzeltmek

Gerçek sistemlerde en değerli özelliklerden biri budur.

Örneğin model şu sorguyu üretmiş olsun:

```sql
SELECT Artist.Name
FROM Artist
JOIN Track
ON Artist.ArtistId = Track.ArtistId
```

**Hata:**

*no such column: Track.ArtistId*

**Bu durumda Agent:**

1. Hatayı okur.
2. Şemayı tekrar inceler.
3. Doğru ilişkiyi bulur.
4. Yeni SQL üretir.

**Doğru sorgu:**

```sql
SELECT Artist.Name
FROM Artist
JOIN Album
ON Artist.ArtistId = Album.ArtistId
JOIN Track
ON Album.AlbumId = Track.AlbumId
```

Bu yaklaşımın adı:

**Self-Correction Pattern**

olarak geçer.

Bu çalışma dosyası Medeniyet Üniversitesi Teknopark Doğal Dil İşleme ve Büyük Dil Modeleri Dersleri için [Yavuz Kömeçoğlu](https://yavuzkomecoglu.com/) tarafından https://www.langchain.com/blog/llms-and-sql referans alınarak hazırlanmıştır.


